# 双鞭毛虫 v2：阻尼运动、开放水域、梯度训练
先在 Colab 运行时设置中启用 GPU，再顺序运行。模型仍以 100 tick/s 接收左右浓度传感器；训练损失才可读取资源位置。

In [ ]:
!if [ -d /content/Parrot/.git ]; then git -C /content/Parrot pull --ff-only; else git clone -q https://github.com/DoctorAwe/Parrot.git /content/Parrot; fi
%cd /content/Parrot
!pip -q install numpy gradio plotly
import torch
assert torch.cuda.is_available(), '请在运行时设置中启用 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
from google.colab import drive
drive_ok = False
try:
    drive.mount('/content/drive', timeout_ms=30000)
    drive_ok = Path('/content/drive/MyDrive').is_dir()
except (ValueError, OSError, TimeoutError) as exc:
    print('Drive 不可用，结果将保存到 Colab 本地：', exc)
out = (Path('/content/drive/MyDrive/Parrot/proto_gradient_v2') if drive_ok
       else Path('/content/proto_gradient_v2'))
out.mkdir(parents=True, exist_ok=True)
from proto_microbe import ProtoConfig, BilateralNerveNet, ProtoNutrientWorld, rollout, replay_figure
from proto_cuda import evaluate_cuda
from proto_gradient import train_gradient
cfg = ProtoConfig()
net = BilateralNerveNet(cfg, seed=404)
print(cfg)
print('Neurons:', net.n, 'synapses:', net.synapses, 'result folder:', out)

## 1. 运动检查
新世界宽 60，资源点 144 个；运动空间不设反弹墙。鞭毛推进可压到近零，身体速度受阻尼。下例给两根鞭毛持续制动信号，观察速度衰减。

In [ ]:
import numpy as np
world = ProtoNutrientWorld(3000, cfg)
for _ in range(300):
    world.step(np.array([-2., 2., -2., 2.]))
print('制动 3 秒后的速度:', round(world.speed, 5), '最大速度:', cfg.max_speed)
print('基线单世界吃到资源:', rollout(net, net.reflex_genome(), 3000))

## 2. 分段梯度训练
每个世界逐 tick 前进，每 100 tick 反传一次。损失来自当前位置到尚未吃掉的最近资源的平滑距离；没有逐 tick 正确动作标签。训练前四分之一阶段只更新传感器到运动神经元的直接连接，然后开放所有突触。

In [ ]:
best, history = train_gradient(net, out=out, epochs=16, worlds=16,
                               ticks_per_episode=600, segment=100,
                               learning_rate=0.015, device='cuda')
print('训练完成，权重已保存:', out / 'best_genome.npy')

## 3. 独立测试
用 30 秒、未见过的种子和真实捕获数比较训练前后。训练中显示的距离损失不能代替此评价。

In [ ]:
import json
test_seeds = range(3000, 3020)
results = {
    'reflex': evaluate_cuda(net, net.reflex_genome(), test_seeds),
    'gradient': evaluate_cuda(net, best, test_seeds),
}
print(json.dumps(results, indent=2))
(out / 'test_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')

## 4. 重播与外链 Web 演示
轨迹只供观察，不作为神经模型输入。Web 演示链接只在 Colab 运行时活动期间有效。

In [ ]:
score, replay = rollout(net, best, 3000, record=True)
(out / 'replay_seed3000.json').write_text(json.dumps(replay), encoding='utf-8')
replay_figure(replay, every=30).show()

In [ ]:
if not drive_ok:
    import shutil
    from google.colab import files
    archive = shutil.make_archive('/content/proto_gradient_v2_results', 'zip', root_dir=out)
    files.download(archive)
else:
    print('结果已保存到:', out)

In [ ]:
from proto_web import create_demo
from IPython.display import HTML, display
demo = create_demo(net, best)
demo.launch(share=True, inline=False, show_error=True)
if demo.share_url:
    display(HTML(f'<a href="{demo.share_url}" target="_blank" rel="noopener">在新标签页打开 Web 演示</a>'))
    print('External demo:', demo.share_url)